# 🛰️ TUA SOPRANOS — ML Eğitim Notebook'u

**Bu notebook:**
1. GitHub'dan projeyi indirir
2. Gerekli kütüphaneleri kurar
3. Space-Track'ten veri çeker
4. **CUDA ile** XGBoost + LSTM eğitir
5. Eğitilmiş modelleri Google Drive'a kaydeder

⚠️ Çalıştırmadan önce: `Runtime → Change runtime type → T4 GPU` seç!


## Hücre 1 — GPU Kontrol

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA mevcut: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU yok! Runtime > Change runtime type > T4 GPU seçin')

## Hücre 2 — Google Drive Bağla (modelleri kaydetmek için)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/TUA_SOPRANOS_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Drive klasörü: {DRIVE_SAVE_DIR}')

## Hücre 3 — Kütüphaneleri Kur

In [ ]:
!pip install sgp4 xgboost scikit-learn requests -q
print('✅ Kütüphaneler kuruldu')

## Hücre 4 — Projeyi GitHub'dan Clone Et

In [ ]:
GITHUB_REPO = 'https://github.com/ozaninaltekin01/TUA_SOPRANOS.git'
BRANCH      = 'claude/focused-hodgkin'

# Önceki çalıştırmadan kalan klasörleri temizle
!rm -rf /content/repo

# Repo'yu clone et — tua_sopranos1/ doğrudan /content/tua_sopranos1 olarak gelir
!git clone --branch {BRANCH} --single-branch {GITHUB_REPO} /content/repo
!cp -r /content/repo/tua_sopranos1/. /content/tua_sopranos1

# Kontrol
print("\n✅ Hazır — /content/tua_sopranos1 içeriği:")
!ls /content/tua_sopranos1

In [ ]:
# İkinci kez çalıştırıyorsan (repo zaten varsa) sadece bunu çalıştır:
# !git -C /content/repo pull origin claude/focused-hodgkin
# !cp -r /content/repo/tua_sopranos1 /content/tua_sopranos1

## Hücre 5 — Python Path Ayarla

In [ ]:
import sys
import os

PROJECT_DIR = '/content/tua_sopranos1'
VERI_DIR    = os.path.join(PROJECT_DIR, 'Veri_analizi')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'model')
CACHE_DIR   = os.path.join(PROJECT_DIR, 'cache')

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, VERI_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

print(f'Çalışma dizini: {os.getcwd()}')
print(f'Python path: {sys.path[:3]}')

# Modül erişimini test et
try:
    from Veri_analizi.config import TURKISH_SATELLITES
    print(f'✅ K1 modülleri erişilebilir — {len(TURKISH_SATELLITES)} uydu')
except ImportError as e:
    print(f'❌ K1 import hatası: {e}')
    print('   Dosya yapısını kontrol edin: /content/tua_sopranos1/Veri_analizi/')

try:
    from model.cara_engine import compute_pc
    print('✅ K2 modülleri erişilebilir')
except ImportError as e:
    print(f'❌ K2 import hatası: {e}')

## Hücre 6 — Space-Track Bağlantısını Test Et

In [ ]:
from Veri_analizi.data_fetch import SpaceTrackClient

client = SpaceTrackClient()
ok = client.login()

if ok:
    print('✅ Space-Track bağlantısı başarılı!')
else:
    print('❌ Space-Track bağlantısı başarısız!')
    print('   config.py içindeki kullanıcı adı/şifreyi kontrol edin')

## Hücre 7 — Veri Üret (Cache'e Yaz)

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, VERI_DIR)

from model.ml_training import ConjunctionDatasetGenerator
import numpy as np

print('Veri üretimi başlıyor...')
print('Bu işlem ~5-10 dakika sürebilir (Space-Track API rate limit nedeniyle)')
print()

gen  = ConjunctionDatasetGenerator(use_cache=False)  # Taze veri çek
X, y = gen.generate_dataset(
    target_samples=5000,
    use_socrates=True,
)

print(f'\n✅ Dataset hazır: {X.shape}')
print(f'   GREEN:  {np.sum(y==0)}')
print(f'   YELLOW: {np.sum(y==1)}')
print(f'   RED:    {np.sum(y==2)}')

## Hücre 8 — XGBoost Eğit

In [ ]:
from model.ml_training import train_xgboost
import os

XGB_SAVE = os.path.join(MODEL_DIR, 'xgboost_risk_model.pkl')

print('XGBoost eğitimi başlıyor...')
xgb_result = train_xgboost(X, y, save_path=XGB_SAVE, n_folds=5)

print(f'\n✅ XGBoost eğitimi tamamlandı!')
print(f'   CV Accuracy: {xgb_result["cv_mean"]*100:.2f}% ± {xgb_result["cv_std"]*100:.2f}%')
print(f'   Kaydedildi: {XGB_SAVE}')

## Hücre 9 — LSTM Eğit (CUDA ile)

En uzun adım — ~15-30 dakika (200 epoch, T4 GPU ile hızlı)

In [ ]:
import torch
print(f'Cihaz: {"CUDA - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

from model.ml_training import train_lstm
import os

LSTM_SAVE   = os.path.join(MODEL_DIR, 'lstm_orbit_model.pt')
SCALER_SAVE = os.path.join(MODEL_DIR, 'lstm_scaler.pkl')

print('LSTM eğitimi başlıyor...')
lstm_result = train_lstm(
    save_path=LSTM_SAVE,
    scaler_path=SCALER_SAVE,
    epochs=200,
    batch_size=128,   # GPU'da daha büyük batch kullan
    patience=20,
    val_split=0.2,
)

print(f'\n✅ LSTM eğitimi tamamlandı!')
if lstm_result:
    print(f'   Best val loss:  {lstm_result["best_val_loss"]:.6f}')
    print(f'   Epoch sayısı:   {lstm_result["epochs_trained"]}')

## Hücre 10 — Eğitim Kayıp Grafiği

In [ ]:
import matplotlib.pyplot as plt

if lstm_result and 'train_history' in lstm_result:
    train_h = lstm_result['train_history']
    val_h   = lstm_result['val_history']

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_h, label='Train Loss', color='steelblue', linewidth=1.5)
    ax.plot(val_h,   label='Val Loss',   color='tomato',    linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('LSTM Eğitim Süreci — TUA SOPRANOS')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'lstm_training_curve.png'), dpi=150)
    plt.show()
    print('Grafik kaydedildi: lstm_training_curve.png')
else:
    print('LSTM sonucu yok')

## Hücre 11 — Model Değerlendirme

In [ ]:
from model.model_evaluation import run_full_evaluation

report = run_full_evaluation(
    save_json=True,
    save_plots=True,
    use_cache=True,
)

## Hücre 12 — Feature Importance Görselleştir

In [ ]:
from IPython.display import Image
import os

plot_path = os.path.join(MODEL_DIR, 'feature_importance.png')
if os.path.exists(plot_path):
    display(Image(plot_path))
else:
    print('Grafik bulunamadı')

## Hücre 13 — Modelleri Drive'a Kaydet

In [ ]:
import shutil
import os

files_to_save = [
    'xgboost_risk_model.pkl',
    'lstm_orbit_model.pt',
    'lstm_scaler.pkl',
    'evaluation_results.json',
    'feature_importance.png',
    'lstm_training_curve.png',
]

print(f'Drive klasörüne kopyalanıyor: {DRIVE_SAVE_DIR}')
for fname in files_to_save:
    src = os.path.join(MODEL_DIR, fname)
    dst = os.path.join(DRIVE_SAVE_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  ✅ {fname:<40} ({size_kb:.1f} KB)')
    else:
        print(f'  ⚠️  {fname} — bulunamadı (atlandı)')

print(f'\n✅ Tüm modeller Drive\'a kaydedildi!')
print(f'   {DRIVE_SAVE_DIR}')

## Hücre 14 — Modelleri Colab'dan İndir (Drive yoksa)

Drive kullanmak istemiyorsan, doğrudan tarayıcıya indir:

In [ ]:
from google.colab import files
import os

# Önce bir zip oluştur
!cd {MODEL_DIR} && zip -q models_trained.zip \
    xgboost_risk_model.pkl \
    lstm_orbit_model.pt \
    lstm_scaler.pkl \
    evaluation_results.json 2>/dev/null || true

zip_path = os.path.join(MODEL_DIR, 'models_trained.zip')
if os.path.exists(zip_path):
    print('İndirme başlıyor...')
    files.download(zip_path)
else:
    # Tek tek indir
    for fname in ['xgboost_risk_model.pkl', 'lstm_orbit_model.pt', 'lstm_scaler.pkl']:
        fpath = os.path.join(MODEL_DIR, fname)
        if os.path.exists(fpath):
            files.download(fpath)
            print(f'  ⬇️  {fname}')

## Hücre 15 — Hızlı Doğrulama (İndirmeden önce kontrol et)

In [ ]:
from model.model_evaluation import quick_sanity_check

chk = quick_sanity_check()
print(f'XGBoost : {"✅ OK" if chk["xgboost_ok"] else "❌ FAIL"}')
print(f'LSTM    : {"✅ OK" if chk["lstm_ok"]    else "❌ FAIL"}')
print(f'Pipeline: {"✅ OK" if chk["pipeline_ok"] else "❌ FAIL"}')

for k, v in chk['details'].items():
    print(f'  {k}: {v}')